# 02 — Silver Curation

Reads Bronze Delta tables, deduplicates dimension tables, enforces referential integrity on fact tables, and writes the **Silver** layer.

**Prerequisite:** run `01_bronze_ingest` first.

In [ ]:
import pandas as pd

def read_bronze(name):
    return spark.read.format('delta').table(f'bronze_{name.lower()}').toPandas()

def write_silver(name, df):
    spark.createDataFrame(df).write.mode('overwrite').format('delta').saveAsTable(f'silver_{name.lower()}')
    print(f'  silver_{name.lower()}: {len(df):>7} rows')

In [ ]:
# Deduplicate dimension tables
wegvak    = read_bronze('wegvak').drop_duplicates('wegvak_id')
watergang = read_bronze('watergang').drop_duplicates('watergang_id')
write_silver('wegvak', wegvak)
write_silver('watergang', watergang)

In [ ]:
# Traffic fact tables — filter to valid wegvak_ids
for child, fk in [
    ('verkeersbord',  'wegvak_id'),
    ('verkeerslicht', 'wegvak_id'),
    ('parkeerplaats', 'wegvak_id'),
    ('ovhalte',       'wegvak_id'),
    ('verkeersmeting','wegvak_id'),
]:
    df = read_bronze(child)
    df = df[df[fk].isin(wegvak['wegvak_id'])]
    write_silver(child, df)

In [ ]:
# Water fact tables — filter to valid watergang_ids
for child, fk in [
    ('gemaal',                'watergang_id'),
    ('lozingspunt',           'watergang_id'),
    ('waterpeilmeting',       'watergang_id'),
    ('waterkwaliteitsmeting', 'watergang_id'),
]:
    df = read_bronze(child)
    df = df[df[fk].isin(watergang['watergang_id'])]
    write_silver(child, df)

# Rioolstreng has no FK dependency
write_silver('rioolstreng', read_bronze('rioolstreng'))

print('\n✅ Silver layer complete!')